# PagedKV: Qwen2.5-7B / Llama-3.1-8B on Kaggle T4 x2

This notebook runs the resumable LongBench v2 comparison across Full cache, H2O, SnapKV, Quest, ArkVale, RocketKV, FreeKV, and PagedKV. In Kaggle **Notebook options**, enable **Internet** and select **GPU T4 x2** before running any cells. Run one model per notebook/output directory.

In [ ]:
import subprocess

gpu_rows = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'],
    text=True,
).strip().splitlines()
print('\n'.join(gpu_rows))
assert len(gpu_rows) >= 2, 'Select GPU T4 x2 in Kaggle Notebook options.'

## Clone the repository

For a new notebook this clones `main`. Once a large run starts, do not pull or change code until that output directory is complete; resume identity includes the source revision.

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/kaggle/working/PagedKV')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/JayGor-13/PagedKV.git', str(repo)], check=True)
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
print('Source commit:', commit)

## Select one model

Qwen2.5-7B is public and is the default. For Llama-3.1-8B, change `MODEL` and `RUN_TAG`, accept Meta's terms on Hugging Face, and create a private Kaggle secret named `HF_TOKEN`.

In [ ]:
import shlex
from pathlib import Path

MODEL = 'Qwen/Qwen2.5-7B-Instruct'
RUN_TAG = 'qwen7b'
# For Llama use:
# MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
# RUN_TAG = 'llama8b'

env_lines = [
    f'export MODEL={shlex.quote(MODEL)}',
    f'export RUN_TAG={shlex.quote(RUN_TAG)}',
]
if MODEL.startswith('meta-llama/'):
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    env_lines.append(f'export HF_TOKEN={shlex.quote(token)}')
Path('/kaggle/working/pagedkv-large.env').write_text('\n'.join(env_lines) + '\n')
print('Configured:', MODEL, 'as', RUN_TAG)

## Optional: restore a previous session

For a fresh run, leave `RESTORE_ARCHIVE = None`. To resume, attach the checkpoint ZIP to the notebook, set its path below, and run this cell before installation. The cell checks out the exact recorded source commit and restores only `outputs/`.

In [ ]:
import subprocess
import zipfile
from pathlib import Path

RESTORE_ARCHIVE = None
# Example after attaching the ZIP as a Kaggle input:
# RESTORE_ARCHIVE = '/kaggle/input/pagedkv-checkpoints/pagedkv-qwen7b-checkpoints.zip'

if RESTORE_ARCHIVE:
    with zipfile.ZipFile(RESTORE_ARCHIVE) as zf:
        recorded_commit = zf.read('CODE_COMMIT.txt').decode().strip()
        subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', recorded_commit], check=True)
        members = [name for name in zf.namelist() if name.startswith('outputs/')]
        zf.extractall(repo, members=members)
    print('Restored outputs at source commit:', recorded_commit)
else:
    print('Fresh run: no checkpoint archive selected.')

## Install isolated environments

This can take 15–30 minutes. It creates separate environments for the pinned baseline and PagedKV Transformers versions.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/PagedKV
bash scripts/setup_kaggle_t4.sh

## Run regression tests

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/PagedKV
bash scripts/test_kaggle_t4.sh

## Mandatory two-GPU smoke matrix

This runs one short example for every method on both benchmarks. Model downloads make the first run slower. Rerunning this exact cell resumes the same smoke directory.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/PagedKV
source /kaggle/working/pagedkv-large.env
export CUDA_VISIBLE_DEVICES=0,1
PYTHONPATH= .envs/t4-baselines/bin/python -u -m experiments.kaggle_t4 run \
  --gpus 2 --models "$MODEL" --smoke \
  --out "outputs/t4-${RUN_TAG}-smoke"

In [ ]:
import json
from pathlib import Path

report_path = Path(f'/kaggle/working/PagedKV/outputs/t4-{RUN_TAG}-smoke/comparison.json')
report = json.loads(report_path.read_text())
failed = [(r['benchmark'], r['method'], r['status']) for r in report['rows'] if r['status'] != 'completed']
print('Incomplete or failed:', failed)
assert not failed, 'Open each failed method result.log before starting the full run.'

## Full LongBench v2 run: 503 examples × 8 methods

This is the main Kaggle experiment. It evaluates every LongBench v2 example with a 16,384-token prompt cap and 128 generated tokens. It is likely to span multiple Kaggle sessions. Repeat this exact cell with the restored output directory to resume; completed examples are skipped.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/PagedKV
source /kaggle/working/pagedkv-large.env
export CUDA_VISIBLE_DEVICES=0,1
PYTHONPATH= .envs/t4-baselines/bin/python -u -m experiments.kaggle_t4 run \
  --gpus 2 --models "$MODEL" \
  --benchmarks longbenchv2 --samples 503 --selection stratified \
  --prompt-cap 16384 --max-new-tokens 128 \
  --out "outputs/t4-${RUN_TAG}-longbench-full503"

## Inspect the combined results

Run this after the launcher finishes. The result is complete only when every row reports 503 samples and `completed`.

In [ ]:
import pandas as pd
from IPython.display import display

csv_path = f'/kaggle/working/PagedKV/outputs/t4-{RUN_TAG}-longbench-full503/comparison.csv'
results = pd.read_csv(csv_path)
display(results[[
    'model', 'benchmark', 'method', 'status', 'samples', 'expected',
    'subset_accuracy_pct', 'generated_tokens', 'elapsed_sample_seconds',
    'elapsed_total_seconds', 'effective_output_tokens_per_second',
]])
incomplete = results[(results.status != 'completed') | (results.samples != results.expected)]
print('Incomplete rows:', len(incomplete))
display(incomplete)

## Save checkpoints before ending a Kaggle session

Stop the running shell cell cleanly, then create this archive. Download it from the displayed link. In a later notebook, clone the same source commit, run setup, extract the archive under `/kaggle/working/PagedKV`, recreate the same model configuration, and rerun the full command.

In [ ]:
import subprocess
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

root = Path('/kaggle/working/PagedKV')
archive = Path(f'/kaggle/working/pagedkv-{RUN_TAG}-checkpoints.zip')
wanted = [
    root / 'outputs' / 't4-environment',
    root / 'outputs' / f't4-{RUN_TAG}-smoke',
    root / 'outputs' / f't4-{RUN_TAG}-longbench-full503',
]
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('CODE_COMMIT.txt', subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], cwd=root, text=True))
    for directory in wanted:
        if directory.exists():
            for path in directory.rglob('*'):
                if path.is_file() and path.suffix != '.tmp' and path.name != '.run.lock':
                    zf.write(path, path.relative_to(root))
print('Archive:', archive, 'size:', round(archive.stat().st_size / 2**20, 1), 'MiB')
display(FileLink(str(archive)))

## LongGenBench remains an H200 experiment

The Kaggle runner can perform shortened LongGenBench diagnostics, but final results require 16K-token generation and the Qwen3-32B judge. Run that protocol on the H200 system. Do not combine a shortened T4 diagnostic with the final paper table.